In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

shot_number = 171348
yaml_path = "../src/faith/datasets/prepare/config/raw.yaml"
with open(yaml_path) as f:
    cfg = yaml.safe_load(f)

from fusionaihub.datasets.prepare.data_extraction import (
    align_signal,
    extract_running_time,
    extract_signal,
)
from fusionaihub.datasets.prepare.sample_processing import (
    remove_empty_samples,
    save_sample,
    split_samples,
)
from fusionaihub.datasets.prepare.signal_processing import (
    identity_transform,
    resample_transform,
    stft_transform,
)

In [20]:
start_time, end_time = extract_running_time(
    shot_number=shot_number,
    directory=Path(cfg["raw_data_dir"]),
    ip_threshold=cfg["ip_threshold"],
    start_time=cfg["start_time"],
    end_time=cfg["end_time"],
)

In [21]:
dfs = []
missing_signals = []
for signal in cfg["signal"].items():
    print(signal[0])
    print(signal[1])
    try:
        df = extract_signal(
            shot_number=shot_number,
            directory=Path(cfg["raw_data_dir"]),
            signal=signal[0],
            start_time=start_time,
            end_time=end_time,
        )
        df.columns = [
            f"{signal[1]['abbr']}_{col}" if col != "time" else col
            for col in range(len(df.columns))
        ]

        # # Add a log to validate our assumption about the config and the transform key:
        # logger.debug(f"Signal config for {signal['name']}: {signal}")

        # Add a column to the dataframe for this signal indicating if a transform is present.
        # We'll use the signal's abbreviation to name the column, e.g., 'IP_transform'
        df = align_signal(
            df=df,
            start_time=start_time,
            end_time=end_time,
            fs=cfg["fs_khz"],
        )
        dfs.append(df)
    except Exception:
        for channel in range(int(signal[1]["expected_channels"])):
            missing_signals.append((signal[1]["abbr"], channel))

magnetics_high_resolution
{'abbr': 'mhr', 'make_stft': False, 'expected_channels': 8}


In [22]:
df = pd.concat(dfs, axis=1, join="inner")
for signal_abbr, channel in missing_signals:
    df[f"{signal_abbr}_{channel}"] = np.nan
    df[f"{signal_abbr}_{channel}_state"] = False

In [23]:
samples = split_samples(
    df=df,
    shot_number=shot_number,
    window_ms=cfg["window_ms"],
    hop_ms=cfg["hop_ms"],
    fs_khz=cfg["fs_khz"],
)
len(samples)

22

In [24]:
print([sample.keys() for sample in samples])

[dict_keys(['171348_0']), dict_keys(['171348_1']), dict_keys(['171348_2']), dict_keys(['171348_3']), dict_keys(['171348_4']), dict_keys(['171348_5']), dict_keys(['171348_6']), dict_keys(['171348_7']), dict_keys(['171348_8']), dict_keys(['171348_9']), dict_keys(['171348_10']), dict_keys(['171348_11']), dict_keys(['171348_12']), dict_keys(['171348_13']), dict_keys(['171348_14']), dict_keys(['171348_15']), dict_keys(['171348_16']), dict_keys(['171348_17']), dict_keys(['171348_18']), dict_keys(['171348_19']), dict_keys(['171348_20']), dict_keys(['171348_21'])]


In [25]:
samples = remove_empty_samples(samples)
len(samples)

22

In [26]:
for sample in samples:
    transformed_samples = {}
    for key, value in sample.items():
        for signal in cfg["signal"].items():
            abbr = signal[1]["abbr"]
            cols = [col for col in value.columns if abbr in col]
            transformed_samples[abbr] = identity_transform(
                x=value[cols].to_numpy().T
            )
            print(abbr, transformed_samples[abbr].shape)
        save_sample(transformed_samples, Path("../data"), key)

mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)
mhr (8, 1, 110000)


In [12]:
first_arr = np.array([list(samples[0].values())[0].iloc[:, 0].values])
transform_shape = stft_transform(x=first_arr).shape
print(first_arr.shape, transform_shape)

(1, 110000) (1, 430, 513)


In [19]:
for sample in samples:
    transformed_samples = {}
    for key, value in sample.items():
        for signal in cfg["signal"].items():
            abbr = signal[1]["abbr"]
            cols = [col for col in value.columns if abbr in col]
            if signal[1]["make_stft"]:
                transformed_samples[abbr] = stft_transform(
                    x=value[cols].to_numpy().T,
                    n_fft=cfg["stft"]["n_fft"],
                    hop_length=cfg["stft"]["hop_length"],
                )
            else:
                transformed_samples[abbr] = resample_transform(
                    x=value[cols].to_numpy().T,
                    ref_shape=transform_shape,
                )
            print(abbr, transformed_samples[abbr].shape)
        save_sample(transformed_samples, Path("../data"), key)
        break
    break

mhr (8, 2551, 513)
ece (48, 2551, 513)
co2 (4, 2551, 513)
gas (5, 2551, 1)
ech (11, 2551, 1)
pin (8, 2551, 1)
tin (8, 2551, 1)
